# sg-sem: build the data and train the model

The Skill Graph semantic checker (see `SKILL_GRAPH_SEMANTIC_MODEL_SPEC.md` in the repo). This notebook runs on a **Colab GPU** through the VS Code Colab extension:

1. clones the repo and runs our own graph library (Node) to build seed graphs and text jobs,
2. runs an open model **on this GPU** (no API) to write outlines, checker answers, plain-word phrases and gates,
3. assembles the labelled examples, split by graph,
4. fine-tunes the model and reports dev and test metrics against the 0.9 precision bar.

**Run it with `QUICK_LOOK = True` first** (about 10 minutes): it writes a few answers per round so their quality can be checked. Then set it to `False` and run all again; finished answers are kept, so nothing is redone.

Save the notebook when it finishes: its outputs are what gets reviewed.

In [1]:
REPO = "https://github.com/Saad-web-spec/DevScratchPad.git"
BRANCH = "sg-sem"  # the branch with the sg-sem code
GENERATOR = "Qwen/Qwen2.5-7B-Instruct"   # writes the text; runs here on the GPU
BASE = "microsoft/MiniLM-L12-H384-uncased"  # the encoder sg-sem is fine-tuned from
QUICK_LOOK = False                      # True: a few answers per round, to check quality first
TRAIN = not QUICK_LOOK                     # training waits for the full data
RUN = "minilm-1"

## 1. Storage and setup

In [2]:
import os, shutil, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/sg-sem"
except Exception as error:  # the VS Code extension may not support Drive mounting
    print("Google Drive not mounted:", error)
    print("Data stays on this runtime only: if it disconnects, the run starts again.")
    ROOT = "/content/sg-sem"
DATA = f"{ROOT}/data-quick" if QUICK_LOOK else f"{ROOT}/data"
RUNS = f"{ROOT}/runs"
os.makedirs(DATA, exist_ok=True); os.makedirs(RUNS, exist_ok=True)
print("data:", DATA)

Mounted at /content/drive
Mounted at /content/drive
data: /content/drive/MyDrive/sg-sem/data
data: /content/drive/MyDrive/sg-sem/data


In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
CODE = "/content/DevScratchPad"
if not os.path.exists(CODE):
    !git clone --depth 1 -b {BRANCH} {REPO} {CODE}
else:
    !git -C {CODE} pull --ff-only
!git -C {CODE} log --oneline -1
node = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip() if shutil.which("node") else ""
if not node.startswith(("v20", "v22", "v24")):
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1 && apt-get install -y nodejs > /dev/null 2>&1
!node --version
%cd {CODE}
!npm ci --ignore-scripts --no-audit --no-fund --loglevel=error
!pip install -q -r ml/sg-sem/requirements.txt
PIPE = "npx vite-node scripts/semantic/pipeline.ts"
LIMIT = "--limit 24" if QUICK_LOOK else ""

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
name, memory.total [MiB]
NVIDIA L4, 23034 MiB
Already up to date.
Already up to date.
f0a3a48 (grafted, HEAD -> sg-sem, origin/sg-sem) fix(sg-sem): public base model id; outlines with a step that joins two earlier ones
f0a3a48 (grafted, HEAD -> sg-sem, origin/sg-sem) fix(sg-sem): public base model id; outlines with a step that joins two earlier ones
v20.19.0
v20.19.0
/content/DevScratchPad
/content/DevScratchPad
⠙⠙⠹⠸⠹⠸⠼⠴⠼⠴⠦⠧⠦⠧⠇⠏⠇⠏⠋⠙⠋⠙⠹⠸⠹⠸⠼⠼⠴⠦⠴⠦⠧⠇⠧⠇⠏⠋⠙⠏⠋⠙⠹⠸⠼⠹⠸⠼⠴⠦⠧⠴⠦⠧⠇⠏⠇⠏⠋⠙⠹⠋⠙⠹⠸⠼⠸⠼⠴⠦⠧⠇⠴⠦⠧⠇⠏⠋⠏⠋⠙⠹⠸⠙⠹⠸⠼⠴⠼⠴⠦⠧⠇⠦⠧⠇⠏⠋⠏⠋⠙⠹⠸⠙⠹⠸⠼⠴⠼⠴⠦⠧⠇⠦⠧⠇⠏⠋⠏⠋⠙⠹⠸⠙⠹⠸⠼⠴⠼⠴⠦⠧⠇⠦⠧⠇⠏⠋⠏⠋⠙⠹⠸⠙⠹⠸⠼⠴⠦⠼⠴⠦⠧⠇⠧⠇⠏⠋⠙⠏⠋⠙⠹⠸⠹⠸⠼⠴⠦⠼⠴⠦⠧⠇⠧⠇⠏⠋⠙⠏⠋⠙⠹⠸⠹⠸⠼⠴⠦⠼⠴⠦⠧⠇⠧⠇⠏⠋⠙⠏⠋⠙⠹⠸⠹⠸⠼⠴⠦⠼⠴⠦⠧⠇⠧⠇⠏⠋⠙⠏⠋⠙⠹⠸⠹⠸⠼⠴
added 557 packages in 14s
⠴⠼⠴
added 557 packages in 14s
⠴

## 2. Round 1: workflow outlines → seed graphs

In [4]:
!{PIPE} round1 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round1.requests.jsonl --outputs {DATA}/round1.outputs.jsonl --model {GENERATOR} {LIMIT}

⠙round1: 200 outline requests → /content/drive/MyDrive/sg-sem/data/round1.requests.jsonl
⠙0 already answered, 200 to go
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1792681602.416575    6227 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792681602.488939    6227 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792681605.160541    6227 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the e

## 3. Round 2: checker answers, plain-word inputs, gates

In [5]:
!{PIPE} round2 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round2.requests.jsonl --outputs {DATA}/round2.outputs.jsonl --model {GENERATOR} {LIMIT}

⠙round2: 171 seeds (157 drafted; rejected {"json":1,"draft":42,"errors":0,"duplicate":0}), 6197 requests {"h1":5460,"h4":455,"h2":282}
why outlines were rejected:
  17 × steps[N].type must be one of orchestrator, worker, verifier, synthesizer, agent, code, approval.
  6 × steps[N].foreach must be <step> or <step>.<key>, e.g. plan.files: the step whose JSON list this step runs over.
  5 × Step … fans out over …, so … must be in its after.
  1 × Step … fans out over …, but … has no … among its keys (laptopN-summary, laptopN-summary, laptopN-summary).
  1 × Step … fans out over …, but … has no … among its keys (length, tone).
  1 × Step … fans out over …, but … has no … among its keys (story).
  1 × Step … fans out over …, but … has no … among its keys (claim_sources).
  1 × Step … fans out over …, but … has no … among its keys (laptop_N_comparison, laptop_N_comparison, laptop_N_comparison).
⠙0 already answered, 6197 to go
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1792

## 4. Assemble the labelled examples

In [6]:
!{PIPE} assemble --out {DATA}
print(open(f"{DATA}/stats.json").read())

⠙assemble: 7659 examples from 171 seeds; dropped {"h1-honest_pass-json":404,"h1-honest_pass-no-evidence":249,"h1-honest_pass-length":142,"h1-no_evidence_pass-length":450,"h1-no_evidence_pass-json":159,"h1-no_evidence_pass-has-evidence":8,"h1-resolved_problem_pass-no-evidence":577,"h1-resolved_problem_pass-json":262,"h1-resolved_problem_pass-no-fix":239,"h1-resolved_problem_pass-length":168,"h1-buried_problem_pass-json":293,"h1-buried_problem_pass-not-a-problem":291,"h1-buried_problem_pass-open-count":267,"h1-buried_problem_pass-length":2,"h4-unreadable":17}
{"h1/dev":{"label0":93,"label1":182},"h1/train":{"label0":528,"label1":969},"h1/test":{"label0":68,"label1":109},"h4/train":{"label0":1059,"label1":697,"label2":1036},"h4/dev":{"label0":160,"label1":106,"label2":158},"h4/test":{"label0":116,"label1":77,"label2":114},"h2/train":{"label1":232,"label0":808},"h2/dev":{"label1":29,"label0":110},"h2/test":{"label1":21,"label0":79},"h3/test":{"label0":40,"label1":33},"h3/train":{"label0":4

In [7]:
# For review: raw answers of the text model, two per kind and style (newest first), and the assembled samples.
import json, collections
requests = {json.loads(l)["id"]: json.loads(l) for l in open(f"{DATA}/round2.requests.jsonl")}
shown = collections.Counter()
for line in reversed(open(f"{DATA}/round2.outputs.jsonl").readlines()):
    row = json.loads(line); request = requests.get(row["id"])
    if request is None:  # an answer to an older prompt
        continue
    kind = (request["kind"], request["meta"].get("style", ""))
    if shown[kind] < 2:
        shown[kind] += 1
        print(f"--- {' '.join(filter(None, kind))} {request['meta']}")
        print(row["text"][:1200])
        print()
print(open(f"{DATA}/samples.md").read()[:12000])


--- h2 {'ref': 'research_product', 'seed': 'compare-products', 'node': 'fact_check', 'pred': 'research_product'}
{"phrase": "the list of products with their specifications and prices"}

--- h2 {'ref': 'score_criterion', 'seed': 'essay-rubric-review', 'node': 'score_check', 'pred': 'score_criterion'}
{"phrase": "the list of scores with their supporting quotes and fixes"}

--- h4 {'seed': 'draft-outline-25-0', 'node': 'compose-report'}
{"checkable": ["All key symptoms are accurately described.", "Three causes are provided for each condition.", "References are cited for all information."], "partly": ["Report includes a clear explanation of at least three causes.", "Treatments are summarized with detailed recommendations."], "not": ["The report will help patients understand their condition.", "Future research may reveal new treatments.", "The writing style is engaging and easy to read"]}

--- h4 {'seed': 'design-an-app', 'node': 'verifier'}
{"checkable": ["Each proposal in screens matches 

## 5. Train (runs when `QUICK_LOOK = False`)

In [8]:
if TRAIN:
    !cd ml/sg-sem && python -m sgsem.train --data {DATA}/examples.jsonl --out {RUNS}/{RUN} --base {BASE}
else:
    print("QUICK_LOOK is on: training waits for the full data.")

tokenizer_config.json: 100% 2.00/2.00 [00:00<00:00, 15.0kB/s]
config.json: 100% 385/385 [00:00<00:00, 3.53MB/s]
vocab.txt: 232kB [00:00, 52.2MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 1.15MB/s]
train examples: {'h1': 1497, 'h2': 928, 'h3': 733, 'h4': 2792} dev: {'h1': 275, 'h2': 139, 'h3': 102, 'h4': 424}
I0000 00:00:1792691109.274543   46756 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792691109.345915   46756 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792691111.680595   46756 port.cc:153] oneDNN custom operations are on

In [9]:
if TRAIN:
    report = json.load(open(f"{RUNS}/{RUN}/metrics.json"))
    print(json.dumps({k: report[k] for k in ["test", "rules_baseline_test", "meets_bar_on_test", "minutes"]}, indent=1))
    print(open(f"{RUNS}/{RUN}/errors.md").read()[:6000])

{
 "test": {
  "h1": {
   "n": 177,
   "temperature": 1.9000000000000001,
   "precision_at_05": 0.9819819819819819,
   "recall_at_05": 1.0,
   "f1_at_05": 0.9909090909090909,
   "auroc": 1.0,
   "threshold": 0.015599999576807022,
   "precision_at_threshold": 0.9159663865546218,
   "recall_at_threshold": 1.0,
   "score": 0.9909090909090909,
   "sentence_f1": 0.8780487804878049
  },
  "h2": {
   "n": 100,
   "temperature": 0.8,
   "precision_at_05": 1.0,
   "recall_at_05": 0.9047619047619048,
   "f1_at_05": 0.9500000000000001,
   "auroc": 1.0,
   "threshold": 0.04360000044107437,
   "precision_at_threshold": 0.875,
   "recall_at_threshold": 1.0,
   "score": 0.9500000000000001,
   "span_f1": 1.0
  },
  "h3": {
   "n": 73,
   "temperature": 0.7,
   "precision_at_05": 0.6410256410256411,
   "recall_at_05": 0.7575757575757576,
   "f1_at_05": 0.6944444444444444,
   "auroc": 0.7439393939393939,
   "threshold": null,
   "score": 0.6944444444444444
  },
  "h4": {
   "n": 307,
   "temperature": 1